# Lesson 9B: PPO Practical - Implementation and Training

## Introduction

Implement PPO from scratch in PyTorch and train on continuous control tasks (HalfCheetah, Hopper). Compare with Stable-Baselines3 for validation.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: PPO Networks and Agent

In [ ]:
class PPOActor(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, state):
        features = self.net(state)
        mean = torch.tanh(self.mean(features))
        std = torch.exp(self.log_std.clamp(-20, 2))
        return mean, std

class PPOCritic(nn.Module):
    def __init__(self, state_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state):
        return self.net(state).squeeze(-1)

print("PPO networks: defined")

## Part 2: PPO Agent with Clipped Objective

In [ ]:
class PPOAgent:
    def __init__(self, state_dim: int, action_dim: int, lr: float = 3e-4, 
                 clip_eps: float = 0.2, entropy_coeff: float = 0.01):
        self.actor = PPOActor(state_dim, action_dim).to(device)
        self.critic = PPOCritic(state_dim).to(device)
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr)
        self.clip_eps = clip_eps
        self.entropy_coeff = entropy_coeff
        self.action_dim = action_dim
    
    def select_action(self, state: np.ndarray):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            mean, std = self.actor(state_t)
            dist = torch.distributions.Normal(mean, std)
            action = dist.sample()
        return action.cpu().numpy()[0], dist.log_prob(action).cpu().numpy()[0]
    
    def update(self, states, actions, log_probs_old, returns, advantages, epochs: int = 4, batch_size: int = 64):
        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.FloatTensor(actions).to(device)
        log_probs_old_t = torch.FloatTensor(log_probs_old).to(device)
        returns_t = torch.FloatTensor(returns).to(device)
        advantages_t = torch.FloatTensor(advantages).to(device)
        
        for epoch in range(epochs):
            indices = np.arange(len(states))
            np.random.shuffle(indices)
            
            for i in range(0, len(states), batch_size):
                batch_idx = indices[i:i+batch_size]
                
                states_batch = states_t[batch_idx]
                actions_batch = actions_t[batch_idx]
                log_probs_old_batch = log_probs_old_t[batch_idx]
                returns_batch = returns_t[batch_idx]
                advantages_batch = advantages_t[batch_idx]
                
                # Actor update with PPO clipping
                mean, std = self.actor(states_batch)
                dist = torch.distributions.Normal(mean, std)
                log_probs = dist.log_prob(actions_batch).sum(dim=1)
                
                ratio = torch.exp(log_probs - log_probs_old_batch)
                clipped_ratio = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps)
                
                actor_loss = -torch.min(ratio * advantages_batch, clipped_ratio * advantages_batch).mean()
                entropy = dist.entropy().sum(dim=1).mean()
                actor_loss = actor_loss - self.entropy_coeff * entropy
                
                self.actor_opt.zero_grad()
                actor_loss.backward()
                self.actor_opt.step()
                
                # Critic update
                values = self.critic(states_batch)
                critic_loss = ((values - returns_batch) ** 2).mean()
                
                self.critic_opt.zero_grad()
                critic_loss.backward()
                self.critic_opt.step()
        
        return actor_loss.item(), critic_loss.item()

print("PPO Agent: defined")

## Part 3: Generalized Advantage Estimation (GAE)

In [ ]:
def compute_gae(values, rewards, dones, gamma=0.99, lambda_=0.95):
    """
    Compute Generalized Advantage Estimation.
    Balances bias-variance tradeoff in advantage estimation.
    """
    advantages = np.zeros_like(rewards)
    returns = np.zeros_like(rewards)
    gae = 0
    next_value = 0
    
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0 if dones[t] else values[t]
        else:
            next_value = values[t + 1]
        
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * lambda_ * gae
        advantages[t] = gae
        returns[t] = gae + values[t]
    
    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns

print("GAE: defined")

## Part 4: Training on HalfCheetah

In [ ]:
# Quick training for demonstration
env = gym.make('HalfCheetah-v4')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = PPOAgent(state_dim, action_dim, lr=3e-4)

episode_returns = []
steps_collected = 0
timesteps_per_batch = 2048

for episode in range(20):  # Quick demo
    state, _ = env.reset()
    done = False
    episode_return = 0
    
    states, actions, rewards, values, log_probs, dones = [], [], [], [], [], []
    
    while not done:
        action, log_prob = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        with torch.no_grad():
            value = agent.critic(torch.FloatTensor(state).unsqueeze(0).to(device)).cpu().item()
        
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        values.append(value)
        log_probs.append(log_prob)
        dones.append(done)
        
        episode_return += reward
        state = next_state
        steps_collected += 1
    
    episode_returns.append(episode_return)
    
    if steps_collected >= timesteps_per_batch:
        advantages, returns = compute_gae(values, rewards, dones)
        agent.update(states, actions, log_probs, returns.tolist(), advantages.tolist())
        steps_collected = 0
    
    if (episode + 1) % 5 == 0:
        print(f"Episode {episode + 1}: Return = {episode_return:.2f}")

env.close()
print(f"\nTraining complete. Avg return: {np.mean(episode_returns):.2f}")

## Part 5: Stable-Baselines3 Comparison

In [ ]:
try:
    from stable_baselines3 import PPO
    
    env = gym.make('HalfCheetah-v4')
    model = PPO('MlpPolicy', env, verbose=0, learning_rate=3e-4)
    model.learn(total_timesteps=10000)
    
    # Evaluate
    returns_sb3 = []
    for _ in range(3):
        obs, _ = env.reset()
        done = False
        ret = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ret += reward
        returns_sb3.append(ret)
    
    env.close()
    print(f"Stable-Baselines3 PPO avg return: {np.mean(returns_sb3):.2f}")
except ImportError:
    print("Stable-Baselines3 not available. Install: pip install stable-baselines3")

## Summary

1. **PPO clipping**: Simple, effective trust region mechanism
2. **Actor-Critic architecture**: Separate policy and value networks
3. **GAE**: Balances variance and bias in advantage estimation
4. **Multiple epochs**: Reuse batch data for sample efficiency
5. **Continuous control**: PPO handles action distributions naturally

PPO is the workhorse of modern RL — used in robotics, games (DOTA2, Minecraft), and reinforcement learning research.